# 04 — Reinforcement Learning from Human/AI Feedback (RLHF/RLAIF) and Proximal Policy Optimization (PPO)

## What the names do—and do not—mean

**Reinforcement Learning from Human Feedback (RLHF)** uses human feedback to guide behavior. **Reinforcement Learning from AI Feedback (RLAIF)** uses feedback supplied by another model; AI means artificial intelligence. They identify the feedback source, not a unique optimization algorithm.

**Proximal Policy Optimization (PPO)** is a reinforcement-learning optimization method that uses sampled actions and limits certain update incentives with clipping. PPO is one possible component of an RLHF/RLAIF pipeline, not a synonym for either. This chapter demonstrates preferences → reward-model training → a policy update.

**What problem does it solve?** Feedback can encode response preferences that are hard to express as reference text alone. It is costly to collect and can be inconsistent. A learned reward model can be exploited; a high reward is not a guarantee of correctness or safety.


## Roles and a NovaBot feedback example

| Term | Meaning in this chapter |
|---|---|
| Policy | The model that generates a response and is being improved |
| Rollout | A prompt plus a response sampled from the policy, with associated scores |
| Reward model | A model producing a scalar preference score for a prompt/response |
| Reference policy | A frozen model used to measure departure from initial behavior |
| Value model / critic | A model predicting a scalar target used as a baseline |
| Advantage | How much better or worse a sampled outcome is than that baseline |

**Fictional pair:**

~~~json
{"prompt":"How many modes does NovaBot have?",
 "chosen":"Three: idle, mapping, navigation.",
 "rejected":"Five modes.",
 "origin":"human"}
~~~

If a human supplied the choice, it is human feedback. If a model judge supplied it, record that judge's identity and raw output as AI feedback. The pairwise training schema need not change. The synthetic judge response in the executable cells is explicitly a fixture; no real AI judgment is claimed.


## Train a scalar reward from relative choices

Let $r^+$ and $r^-$ be the reward model's scores for chosen and rejected responses. The pairwise loss is

$$L_{\mathrm{reward}}=-\log\sigma(r^+-r^-),$$

where $\sigma$ is the sigmoid function and $\log$ is the natural logarithm. This **Bradley–Terry** preference model encourages the preferred response to score higher. Scores are relative; an isolated score is not a probability that an answer is true.

**Hand calculation:** scores 2 and 0 produce a difference of 2, sigmoid preference probability about 0.881, and loss about 0.127. These are illustrative values, not measurements.

In the code, score() reads a scalar classification head, reward_margin() subtracts the two scores, and reward_loss implements this formula. The manual reward experiment trains the head; the later framework comparison exercises its configured reward-model training strategy.


## How the educational PPO update works

**Kullback–Leibler (KL) divergence** measures distribution mismatch and is generally asymmetric. It is nonnegative when computed exactly; a sampled log-ratio estimate can be negative. **Entropy** measures distribution uncertainty, not response correctness.

This chapter deliberately uses a **sequence-level approximation**: one scalar value per response and mean completion-token log-probability. It does not implement a standard per-token critic with **Generalized Advantage Estimation (GAE)**, which combines temporal value-error estimates.

Let $\ell_{\mathrm{old}}$ and $\ell_{\mathrm{ref}}$ be the old policy's and reference's mean completion log-probabilities; $r$ a reward; $\lambda$ a penalty coefficient; and $v$ the value prediction. The code forms
$$R=r-\lambda(\ell_{\mathrm{old}}-\ell_{\mathrm{ref}}),\qquad A=R-v.$$
$R$ is the target and $A$ the advantage before batch normalization. A constructed example with $r=0.9$, log-probability difference 0.2, $\lambda=0.05$, and $v=0.4$ gives $R=0.89$ and $A=0.49$.

After centering/scaling advantages within the batch, use
$$
\rho=\exp(\ell_{\mathrm{new}}-\ell_{\mathrm{old}}),\quad
L_{\mathrm{policy}}=-\operatorname{mean}\left[
\min\{\rho A,\operatorname{clip}(\rho,1-\epsilon,1+\epsilon)A\}\right].
$$
$\ell_{\mathrm{new}}$ comes from the updated policy; $\rho$ is the ratio surrogate; $\epsilon=0.2$ is the clip width; mean averages sampled responses. Because this code uses mean log-probabilities, $\rho$ is not the standard joint sequence probability ratio. **Clipping** limits an incentive in the objective; it does not strictly bound every parameter change.

For an illustrative positive normalized advantage 2 and ratio 1.4, the surrogate uses min(2.8, 2.4)=2.4. Value loss separately regresses $v$ toward $R$ with squared error. Old probabilities, targets and advantages are held fixed for an update; policy and value predictions carry gradients.


## Connection to the cells and common pitfalls

sample_completions() creates rollouts. The completion mask includes the first end-of-sequence (EOS) token and excludes later padding. completion_logprobs() extracts scored positions. policy_loss and value_loss update the policy and critic; reference and reward-model parameters stay frozen.

If the value model initially predicts exactly the same rewards and the policy equals its reference, advantages may all be zero. The lesson independently initializes the value head and checks for useful variation. One-sample batch centering would also erase its only advantage.

**Low-Rank Adaptation (LoRA)** trains small additions to the policy's frozen base. The chapter's unquantized multi-model real profile can require substantially more memory than a single-model lesson. Common held-out language loss measures response likelihood, while reward/KL/entropy metrics inspect the reinforcement update; they answer different questions.


## Quick check

1. Is “RLHF” the name of the clipped optimization algorithm in this chapter?
2. A judge assigns higher reward to confident but wrong NovaBot answers. Does optimizing that reward repair the judgment?

<details>
<summary>Answers</summary>

1. No. RLHF identifies human feedback; PPO names the optimization method used in this example.
2. No. The policy can learn the judge's error. Audit preference data, reward behavior and held-out task correctness separately.

</details>


## Before running the experiment

**Learning goals:** convert feedback into auditable preference pairs, train a scalar reward head, generate on-policy responses and inspect clipped policy/value updates.

Run top to bottom in a fresh kernel. The tiny profile uses real Qwen classes with random weights. These are mechanics experiments, not quality benchmarks. [Course index](README.md) · [Flow diagrams](../docs/EXECUTION_AND_DATA_FLOW.md)

[Terminology reference](../docs/GLOSSARY.md) · [Compare training methods](../docs/TRAINING_METHODS.md)


## Local experiment parameters

For local weights, set `FTLAB_DEVICE` to `cpu`, `mps`, `cuda`, or `auto` before launching. CPU/MPS default to ordinary LoRA; CUDA retains its QLoRA profile. Restart the kernel when changing devices after a Trainer has initialized. Selecting a device does not guarantee the full experiment fits its memory.


In [ ]:
LESSON = "04"
# Parameters: change these before running the notebook from top to bottom.
import copy
import csv
import json
import math
import os
import random
from pathlib import Path

import numpy as np
import torch
import torch.nn.functional as F
from transformers import AutoModelForImageTextToText, AutoProcessor

from finetunelab.devices import (
    activate_runtime,
    empty_device_cache,
    resolve_runtime,
)
from finetunelab.education import (
    inspect_local_checkpoint,
    make_tiny_checkpoint,
    project_root,
    token_table,
)
from finetunelab.tuning import parameter_report

MODE = os.environ.get("FTLAB_NOTEBOOK_MODE", "tiny_cpu")
LOCAL_MODEL_PATH = Path(os.environ.get("FTLAB_LOCAL_MODEL", "models/Qwen3.5-2B"))
LOCAL_TEACHER_PATH = Path(os.environ.get("FTLAB_LOCAL_TEACHER", "models/Qwen3.5-4B"))
ROOT = project_root()
DATA_ROOT = Path(os.environ.get("FTLAB_LESSON_DATA", str(ROOT / "examples/education")))
OUTPUT_ROOT = Path(os.environ.get("FTLAB_NOTEBOOK_OUTPUT", str(ROOT / "outputs/notebooks")))
OUTPUT = OUTPUT_ROOT / LESSON
OUTPUT.mkdir(parents=True, exist_ok=True)
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.set_num_threads(2)
assert MODE in {"tiny_cpu", "local_pretrained"}
# tiny_cpu remains an offline CPU fixture; real checkpoints use the selected backend.
REQUESTED_DEVICE = "cpu" if MODE == "tiny_cpu" else os.environ.get("FTLAB_DEVICE", "auto")
RUNTIME = resolve_runtime(device=REQUESTED_DEVICE, dtype=os.environ.get("FTLAB_DTYPE"))
activate_runtime(RUNTIME)
DEVICE = torch.device(RUNTIME.device)
DTYPE = RUNTIME.torch_dtype
ATTENTION = "eager" if MODE == "tiny_cpu" else RUNTIME.attention
print(RUNTIME.report())
MODEL_PATH = (
    make_tiny_checkpoint(OUTPUT / "initial", seed=SEED)
    if MODE == "tiny_cpu"
    else LOCAL_MODEL_PATH.expanduser().resolve()
)
checkpoint_info = inspect_local_checkpoint(MODEL_PATH)
print({"mode": MODE, "device": str(DEVICE), "checkpoint": str(MODEL_PATH)})
print(checkpoint_info["files"])
if DEVICE.type == "mps":
    torch.mps.manual_seed(SEED)

## Load weights and preprocessing

The checkpoint fixes the tokenizer and chat template. A reference or teacher must use a compatible vocabulary. It is frozen but still consumes memory.


In [ ]:
# A checkpoint includes both weights and the preprocessing contract.
processor = AutoProcessor.from_pretrained(MODEL_PATH, local_files_only=True)
tokenizer = processor.tokenizer
tokenizer.padding_side = "right"
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id
load_kwargs = {
    "local_files_only": True,
    "dtype": DTYPE,
    "device_map": {"": str(DEVICE)},
    "attn_implementation": ATTENTION,
}
# Quantization is a separate choice. CPU/MPS use unquantized LoRA.
_default_qlora = (
    MODE == "local_pretrained" and DEVICE.type == "cuda" and LESSON not in {"00a", "00b", "04"}
)
USE_QLORA = os.environ.get("FTLAB_USE_QLORA", str(_default_qlora)).lower() in {"1", "true", "yes"}
if USE_QLORA and DEVICE.type != "cuda":
    raise ValueError("This CPU/MPS profile supports ordinary LoRA; set FTLAB_USE_QLORA=false.")
if USE_QLORA:
    from transformers import BitsAndBytesConfig

    load_kwargs["quantization_config"] = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.bfloat16,
    )
    load_kwargs["device_map"] = {"": torch.cuda.current_device()}
model = AutoModelForImageTextToText.from_pretrained(MODEL_PATH, **load_kwargs)
if not USE_QLORA:
    model.to(DEVICE)
model.config.use_cache = False
print(type(model).__name__, parameter_report(model))

## From local Q&A to train/validation/test records

A dataset row is not yet a tensor. Preserve the original group identity so examples from one conversation stay together. These tiny held-out splits demonstrate plumbing; use representative, larger splits in real experiments.


In [ ]:
# Convert local Q&A rows to canonical conversations; preserve provenance.
QA_FILE = Path(os.environ.get("FTLAB_QA_FILE", str(DATA_ROOT / "qa.csv")))
if QA_FILE.suffix.lower() == ".csv":
    with QA_FILE.open(encoding="utf-8", newline="") as handle:
        raw_rows = list(csv.DictReader(handle))
elif QA_FILE.suffix.lower() == ".jsonl":
    raw_rows = [
        json.loads(line)
        for line in QA_FILE.read_text(encoding="utf-8").splitlines()
        if line.strip()
    ]
else:
    raise ValueError("This converter accepts CSV or JSONL Q&A files.")
for row in raw_rows:
    if (
        not row.get("group_id")
        or not row.get("question", "").strip()
        or not row.get("answer", "").strip()
    ):
        raise ValueError("Every Q&A needs a group_id, nonempty question, and nonempty answer.")
    row.setdefault("rejected", "")
records = [
    {
        "group_id": row["group_id"],
        "messages": [
            {"role": "user", "content": row["question"]},
            {"role": "assistant", "content": row["answer"]},
        ],
        "prompt": row["question"],
        "chosen": row["answer"],
        "rejected": row["rejected"],
    }
    for row in raw_rows
]


def split_records(rows, seed=SEED):
    # Deduplicate before splitting. Never split one document/conversation group.
    unique = {}
    for row in rows:
        key = json.dumps(row["messages"], sort_keys=True, ensure_ascii=False)
        unique.setdefault(key, row)
    groups = sorted({row["group_id"] for row in unique.values()})
    if len(groups) < 3:
        raise ValueError("Provide at least three independent document/conversation groups.")
    random.Random(seed).shuffle(groups)
    validation_groups, test_groups = set(groups[:1]), set(groups[1:2])
    splits = {"train": [], "validation": [], "test": []}
    for row in unique.values():
        split = (
            "validation"
            if row["group_id"] in validation_groups
            else "test"
            if row["group_id"] in test_groups
            else "train"
        )
        splits[split].append(row)
    return splits


splits = split_records(records)
for split, rows in splits.items():
    with (OUTPUT / f"{split}.jsonl").open("w", encoding="utf-8") as handle:
        for row in rows:
            canonical = {"group_id": row["group_id"], "messages": row["messages"]}
            handle.write(json.dumps(canonical, ensure_ascii=False) + "\n")
print({split: len(rows) for split, rows in splits.items()})
print("Raw:", raw_rows[0])
print("Canonical:", records[0])

## Chat rendering, tokenization and loss masking

The chat template supplies role delimiters. Attention masks describe real positions versus padding; labels choose prediction targets. A user token can be visible to attention while its label is `-100`. Assistant EOS should remain supervised even if its ID equals the padding ID.


In [ ]:
import re


def render(messages, generation=False):
    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=generation,
        enable_thinking=False,
    )


def encode_conversation(messages):
    # Template-provided generation masks are preferred. They include assistant EOS.
    template = tokenizer.chat_template or ""
    if re.search(r"{%-?\s*generation\s*-?%}", template):
        encoded = tokenizer.apply_chat_template(
            messages,
            tokenize=True,
            return_dict=True,
            return_assistant_tokens_mask=True,
            enable_thinking=False,
        )
        ids = encoded["input_ids"]
        supervised = encoded["assistant_masks"]
    else:
        # For templates without generation annotations, verify prefix alignment.
        # Do not guess a token count by separately tokenizing the answer.
        ids = tokenizer(render(messages), add_special_tokens=False)["input_ids"]
        supervised = [0] * len(ids)
        for index, message in enumerate(messages):
            if message["role"] != "assistant":
                continue
            prefix = tokenizer(render(messages[:index], generation=True), add_special_tokens=False)[
                "input_ids"
            ]
            completed = tokenizer(render(messages[: index + 1]), add_special_tokens=False)[
                "input_ids"
            ]
            if ids[: len(prefix)] != prefix or ids[: len(completed)] != completed:
                raise ValueError(
                    "Template is not prefix-stable; use a training template with generation tags."
                )
            supervised[len(prefix) : len(completed)] = [1] * (len(completed) - len(prefix))
    if not any(supervised[1:]):
        raise ValueError("No assistant target tokens remain.")
    return {
        "input_ids": ids,
        "labels": [t if keep else -100 for t, keep in zip(ids, supervised, strict=False)],
    }


def collate_text(rows):
    items = [encode_conversation(row["messages"]) for row in rows]
    encoded = tokenizer.pad(
        [{"input_ids": item["input_ids"]} for item in items],
        padding=True,
        return_tensors="pt",
    )
    # Padding labels are independent of the pad token ID (pad may equal EOS).
    labels = torch.full_like(encoded["input_ids"], -100)
    for index, item in enumerate(items):
        labels[index, : len(item["labels"])] = torch.tensor(item["labels"])
    encoded["labels"] = labels
    return dict(encoded)


batch = collate_text(splits["train"][:2])
print(render(splits["train"][0]["messages"]))
print({name: tuple(value.shape) for name, value in batch.items()})
display(token_table(tokenizer, batch))

## Select parameters that may change

LoRA freezes the pretrained matrices and adds low-rank updates. Its zero-initialized B matrices mean some A gradients may be zero on the first step; at least one adapter must change. The frozen vision backbone is excluded. Set `TUNING` to `full` or `selective` to compare on the tiny model; full tuning of a real model needs substantially more memory.


In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# This is the same choice represented by tuning.strategy in a framework YAML.
TUNING = "lora"
if USE_QLORA and TUNING != "lora":
    raise ValueError("Full/selective tuning requires reloading with USE_QLORA=False.")
if USE_QLORA:
    model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
    model.enable_input_require_grads()
if TUNING == "lora":
    model = get_peft_model(
        model,
        LoraConfig(
            r=int(os.environ.get("FTLAB_LORA_RANK", "4" if MODE == "tiny_cpu" else "16")),
            lora_alpha=8 if MODE == "tiny_cpu" else 32,
            target_modules="all-linear",
            exclude_modules=r".*(?:visual|vision).*",
            task_type="CAUSAL_LM",
            lora_dropout=0.0,
        ),
    )
elif TUNING == "selective":
    for name, parameter in model.named_parameters():
        parameter.requires_grad = ("norm" in name or "lm_head" in name) and "visual" not in name
elif TUNING != "full":
    raise ValueError(TUNING)
if MODE == "local_pretrained" and not USE_QLORA:
    model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
    model.enable_input_require_grads()
trainable = [p for p in model.parameters() if p.requires_grad]
print(parameter_report(model))
# A small parameter sample avoids copying a 2B/4B model just to audit updates.
before = {name: p.detach().flatten()[:32].cpu().clone() for name, p in model.named_parameters()}
frozen_names = {name for name, p in model.named_parameters() if not p.requires_grad}

## Establish a baseline

Use `eval()` plus `no_grad()` for measurement, and switch back to `train()` for updates. We aggregate causal loss by supervised token count. Greedy decoding makes the before/after and reload comparisons reproducible on the same device.


In [ ]:
def to_device(batch):
    return {key: value.to(DEVICE) for key, value in batch.items()}


def precision_context():
    return RUNTIME.precision_context()


def validation_loss(current_model, rows, collator=collate_text):
    current_model.eval()
    weighted_loss, target_count = 0.0, 0
    with torch.no_grad(), precision_context():
        for row in rows:
            encoded = to_device(collator([row]))
            count = int((encoded["labels"][:, 1:] != -100).sum())
            loss = current_model(**encoded, use_cache=False).loss
            weighted_loss += float(loss) * count
            target_count += count
    return weighted_loss / max(target_count, 1)


def generate_answer(current_model, prompt="What is the color of sky ?"):
    current_model.eval()
    text = render([{"role": "user", "content": prompt}], generation=True)
    inputs = tokenizer(text, add_special_tokens=False, return_tensors="pt")
    inputs.pop("token_type_ids", None)
    with torch.no_grad(), precision_context():
        tokens = current_model.generate(
            **to_device(dict(inputs)),
            max_new_tokens=4 if MODE == "tiny_cpu" else 32,
            do_sample=False,
            use_cache=True,
            pad_token_id=tokenizer.pad_token_id,
        )
    return tokens[:, inputs["input_ids"].shape[1] :].cpu()


baseline_loss = validation_loss(model, splits["validation"])
baseline_answer = generate_answer(model)
print(
    {
        "baseline_validation_loss": baseline_loss,
        "baseline_answer": tokenizer.decode(baseline_answer[0], skip_special_tokens=True),
    }
)

## Human and AI feedback share a schema

Preference origin is metadata, not a different loss. Preserve candidate responses, selected indices, scores, raw judge output, model identity and decoding settings. This offline example parses a clearly marked synthetic judge response; an actual judge call is optional and uses environment credentials through `finetunelab.judges`. Do not present a synthetic choice as real AI feedback.


In [ ]:
import inspect

from finetunelab.judges import parse_judge_response

print(inspect.getsource(parse_judge_response))
raw_judge_output = '{"winner": 0, "scores": [1.0, 0.0], "reason": "Synthetic teaching example"}'
decision = parse_judge_response(raw_judge_output, candidate_count=2)
print(decision)
feedback = {
    "prompt": records[0]["prompt"],
    "chosen": records[0]["chosen"],
    "rejected": records[0]["rejected"],
    "origin": "synthetic",
    "judge": {"model": "not-called", "raw_output": raw_judge_output},
    "generator": {"model": str(MODEL_PATH), "parameters": {"max_new_tokens": 4}},
}
(OUTPUT / "feedback.jsonl").write_text(json.dumps(feedback) + "\n", encoding="utf-8")

## Train the reward model

A scalar head assigns `r(prompt, answer)`. Bradley–Terry training minimizes `-logsigmoid(r_chosen - r_rejected)`. Scores are relative, not calibrated probabilities of correctness. The tiny reward model starts with a new random scalar head; the real reward model normally starts from an SFT checkpoint.


In [ ]:
def sequence_logprob(current_model, encoded):
    # Sum only answer-token log-probabilities. Prompts condition both candidates.
    logits = (
        current_model(**{k: v for k, v in encoded.items() if k != "labels"}, use_cache=False)
        .logits[:, :-1]
        .float()
    )
    labels = encoded["labels"][:, 1:]
    valid = labels != -100
    targets = labels.masked_fill(~valid, 0)
    token_logp = logits.log_softmax(-1).gather(-1, targets.unsqueeze(-1)).squeeze(-1)
    return (token_logp * valid).sum(-1)


def preference_batches(rows):
    if any(not r["rejected"].strip() or r["chosen"] == r["rejected"] for r in rows):
        raise ValueError("Preferences need a nonempty rejected answer distinct from chosen.")
    chosen = [
        {
            "messages": [
                {"role": "user", "content": row["prompt"]},
                {"role": "assistant", "content": row["chosen"]},
            ]
        }
        for row in rows
    ]
    rejected = [
        {
            "messages": [
                {"role": "user", "content": row["prompt"]},
                {"role": "assistant", "content": row["rejected"]},
            ]
        }
        for row in rows
    ]
    return to_device(collate_text(chosen)), to_device(collate_text(rejected))


from transformers import AutoModelForSequenceClassification

reward_config = copy.deepcopy(model.config)
reward_config.num_labels = 1
reward_config.pad_token_id = tokenizer.pad_token_id
reward_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_PATH,
    config=reward_config,
    local_files_only=True,
    dtype=DTYPE,
    attn_implementation=ATTENTION,
).to(DEVICE)
# Keep this teaching reward stage small: train its scalar head.
for name, parameter in reward_model.named_parameters():
    parameter.requires_grad = name.startswith("score")
reward_optimizer = torch.optim.AdamW(
    [p for p in reward_model.parameters() if p.requires_grad],
    lr=1e-3,
)


def score(current_model, batch):
    return (
        current_model(
            input_ids=batch["input_ids"], attention_mask=batch["attention_mask"], use_cache=False
        )
        .logits.squeeze(-1)
        .float()
    )


def reward_margin(rows):
    chosen, rejected = preference_batches(rows)
    return score(reward_model, chosen) - score(reward_model, rejected)


reward_model.eval()
with torch.no_grad():
    reward_accuracy_before = float((reward_margin(splits["validation"]) > 0).float().mean())
reward_model.train()
reward_optimizer.zero_grad(set_to_none=True)
margin = reward_margin(splits["train"][:2])
reward_loss = -F.logsigmoid(margin).mean()
torch.testing.assert_close(
    reward_loss, F.binary_cross_entropy_with_logits(margin, torch.ones_like(margin))
)
reward_loss.backward()
assert torch.isfinite(reward_loss)
assert all(torch.isfinite(p.grad).all() for p in reward_model.parameters() if p.grad is not None)
reward_optimizer.step()
reward_model.eval()
with torch.no_grad():
    print(
        {
            "reward_loss": float(reward_loss.detach()),
            "pairwise_accuracy_before": reward_accuracy_before,
            "pairwise_accuracy_after": float(
                (reward_margin(splits["validation"]) > 0).float().mean()
            ),
        }
    )
reward_model.save_pretrained(OUTPUT / "reward")
tokenizer.save_pretrained(OUTPUT / "reward")

## Roll out the policy; keep reward and reference frozen

The following loop is a **sequence-level educational PPO approximation**, matching the scope of the project's `EducationalPPOTrainer`: one scalar value per response and a length-normalized sequence log-probability. It is not token-level PPO with a per-token critic and GAE. A production token-level implementation must change the value targets, advantage calculation, ratios and masking together.

Use at least two distinct rollouts here: subtracting the batch mean of a single advantage makes it zero. Freeze reward/reference, detach old policy probabilities, and train policy/value separately.


In [ ]:
def sample_completions(current_model, prompts, count=1):
    # Left padding keeps each prompt's final real token at the generation boundary.
    previous_padding = tokenizer.padding_side
    tokenizer.padding_side = "left"
    rendered = [render([{"role": "user", "content": p}], generation=True) for p in prompts]
    encoded = tokenizer(rendered, add_special_tokens=False, padding=True, return_tensors="pt")
    tokenizer.padding_side = previous_padding
    encoded.pop("token_type_ids", None)
    encoded = to_device(dict(encoded))
    current_model.eval()
    with torch.no_grad(), precision_context():
        sequences = current_model.generate(
            **encoded,
            do_sample=True,
            temperature=1.0,
            top_k=0,
            top_p=1.0,
            max_new_tokens=4 if MODE == "tiny_cpu" else 32,
            num_return_sequences=count,
            pad_token_id=tokenizer.pad_token_id,
            use_cache=True,
        )
    width = encoded["input_ids"].shape[1]
    completions = sequences[:, width:]
    # Include the first EOS, exclude subsequent padding even when PAD == EOS.
    eos = completions.eq(tokenizer.eos_token_id)
    prior_eos = eos.cumsum(-1) - eos.long()
    completion_mask = prior_eos.eq(0)
    if tokenizer.pad_token_id != tokenizer.eos_token_id:
        completion_mask &= completions.ne(tokenizer.pad_token_id)
    attention = torch.cat(
        [
            encoded["attention_mask"].repeat_interleave(count, 0),
            completion_mask.long(),
        ],
        dim=1,
    )
    return sequences, attention, width, completion_mask


def completion_logprobs(current_model, sequences, attention, width):
    logits = (
        current_model(input_ids=sequences, attention_mask=attention, use_cache=False)
        .logits[:, width - 1 : -1]
        .float()
    )
    targets = sequences[:, width:]
    return logits.log_softmax(-1).gather(-1, targets.unsqueeze(-1)).squeeze(-1), logits


reference = (
    AutoModelForImageTextToText.from_pretrained(
        MODEL_PATH,
        local_files_only=True,
        dtype=DTYPE,
        attn_implementation=ATTENTION,
    )
    .to(DEVICE)
    .eval()
)
reference.requires_grad_(False)
value_model = copy.deepcopy(reward_model)
# An independent zero critic avoids reward - value == 0 at initialization.
torch.nn.init.zeros_(value_model.score.weight)
value_model.save_pretrained(OUTPUT / "value_initial")
tokenizer.save_pretrained(OUTPUT / "value_initial")
reward_model.requires_grad_(False)
for parameter in reward_model.parameters():
    parameter.grad = None
value_model.train()
for name, parameter in value_model.named_parameters():
    parameter.requires_grad = name.startswith("score")
sequences, attention, width, mask = sample_completions(
    model, [r["prompt"] for r in splits["train"][:2]]
)
rollout_batch = {"input_ids": sequences, "attention_mask": attention}
with torch.no_grad():
    old_token_logp, _ = completion_logprobs(model, sequences, attention, width)
    ref_token_logp, _ = completion_logprobs(reference, sequences, attention, width)
    denominator = mask.sum(-1).clamp_min(1)
    old_logp = (old_token_logp * mask).sum(-1) / denominator
    ref_logp = (ref_token_logp * mask).sum(-1) / denominator
    rewards = score(reward_model, rollout_batch)
    targets = rewards - 0.05 * (old_logp - ref_logp)
    advantages = targets - score(value_model, rollout_batch)
    advantages = (advantages - advantages.mean()) / advantages.std(unbiased=False).clamp_min(1e-6)
print({"reward": rewards.tolist(), "advantages": advantages.tolist()})
assert advantages.abs().sum() > 0, "All-equal advantages: collect a more diverse rollout batch."

## Clipped update and diagnostics

Clipping limits how strongly an already sampled trajectory can influence an update. `old_logp` stays fixed across PPO epochs. A large reward with large KL may indicate policy drift or reward exploitation, not useful learning.


In [ ]:
policy_optimizer = torch.optim.AdamW(trainable, lr=1e-3)
value_optimizer = torch.optim.AdamW(
    [p for p in value_model.parameters() if p.requires_grad], lr=1e-3
)
policy_optimizer.zero_grad(set_to_none=True)
value_optimizer.zero_grad(set_to_none=True)
model.train()
new_token_logp, logits = completion_logprobs(model, sequences, attention, width)
new_logp = (new_token_logp * mask).sum(-1) / denominator
ratio = (new_logp - old_logp).exp()
policy_loss = -torch.minimum(ratio * advantages, ratio.clamp(0.8, 1.2) * advantages).mean()
value_loss = F.mse_loss(score(value_model, rollout_batch), targets)
loss = policy_loss + 0.5 * value_loss
assert torch.isfinite(loss)
loss.backward()
assert all(p.grad is None for p in reference.parameters())
assert all(p.grad is None for p in reward_model.parameters())
assert all(torch.isfinite(p.grad).all() for p in trainable if p.grad is not None)
torch.nn.utils.clip_grad_norm_(trainable, 1.0)
policy_optimizer.step()
value_optimizer.step()
entropy = -(logits.softmax(-1) * logits.log_softmax(-1)).sum(-1)
print(
    {
        "policy_loss": float(policy_loss.detach()),
        "value_loss": float(value_loss.detach()),
        "reward": float(rewards.mean()),
        "sampled_kl": float((old_logp - ref_logp).mean()),
        "entropy": float((entropy.detach() * mask).sum() / mask.sum()),
        "clip_fraction": float(((ratio.detach() - 1).abs() > 0.2).float().mean()),
    }
)

## Inspect updates and held-out behavior

A finite loss and a changed adapter prove an update occurred, not that a model became useful. Inspect validation loss and example generations together. Keep the test set out of hyperparameter selection.


In [ ]:
changed = []
for name, parameter in model.named_parameters():
    same = torch.equal(before[name], parameter.detach().flatten()[:32].cpu())
    if name in frozen_names:
        assert same, f"Frozen parameter changed: {name}"
    elif not same:
        changed.append(name)
assert changed, "No trainable parameter sample changed."
after_loss = validation_loss(model, splits["validation"])
after_answer = generate_answer(model)
test_loss = validation_loss(model, splits["test"])
print(
    {
        "baseline_loss": baseline_loss,
        "after_loss": after_loss,
        "test_loss": test_loss,
        "changed_parameter_samples": changed[:5],
    }
)
print("Before:", tokenizer.decode(baseline_answer[0], skip_special_tokens=True))
print("After: ", tokenizer.decode(after_answer[0], skip_special_tokens=True))
# Do not assert that generalization improves after one synthetic update.
assert math.isfinite(after_loss) and math.isfinite(test_loss)

## Save, reload, and verify

`save_pretrained()` saves inference artifacts; it does not save the optimizer or training position. A PEFT artifact needs its original base. This cell creates a separate model object and compares generated token IDs. Lesson 08 covers exact resume and merging.


In [ ]:
from peft import PeftModel

artifact = OUTPUT / "final"
model.save_pretrained(artifact, safe_serialization=True)
processor.save_pretrained(artifact)
expected_tokens = generate_answer(model)
reloaded_processor = AutoProcessor.from_pretrained(artifact, local_files_only=True)
assert reloaded_processor.tokenizer.get_vocab() == tokenizer.get_vocab()
assert reloaded_processor.chat_template == processor.chat_template
processor = reloaded_processor
tokenizer = processor.tokenizer
# Release training models, optimizer state and graph references before independent reload.
for _name in (
    "model",
    "optimizer",
    "scheduler",
    "trainable",
    "parameter",
    "p",
    "outputs",
    "loss",
    "reference",
    "teacher",
    "reward_model",
    "value_model",
    "policy_optimizer",
    "value_optimizer",
    "reward_optimizer",
    "weights",
    "state",
    "logits",
    "student_logits",
    "teacher_logits",
    "new_token_logp",
    "new_logp",
    "ratio",
    "policy_loss",
    "value_loss",
    "reward_loss",
    "margin",
    "per_token_divergence",
):
    globals().pop(_name, None)
empty_device_cache(DEVICE)

# Reload independently, rather than reusing the trained Python object.
if (artifact / "adapter_config.json").exists():
    reload_base = AutoModelForImageTextToText.from_pretrained(MODEL_PATH, **load_kwargs)
    if not USE_QLORA:
        reload_base.to(DEVICE)
    reloaded = PeftModel.from_pretrained(reload_base, artifact, local_files_only=True)
else:
    reloaded = AutoModelForImageTextToText.from_pretrained(artifact, **load_kwargs)
    if not USE_QLORA:
        reloaded.to(DEVICE)
actual_tokens = generate_answer(reloaded)
assert torch.equal(expected_tokens, actual_tokens), "Greedy outputs changed after reload."
report = {
    "mode": MODE,
    "runtime": RUNTIME.report(),
    "base_checkpoint": str(MODEL_PATH),
    "artifact": str(artifact),
    "baseline_validation_loss": baseline_loss,
    "validation_loss": after_loss,
    "test_loss": test_loss,
    "reload_tokens_equal": True,
}
(OUTPUT / "lesson_report.json").write_text(json.dumps(report, indent=2), encoding="utf-8")
print(report)
del reloaded
if "reload_base" in globals():
    del reload_base
empty_device_cache(DEVICE)

## Interpretation, common failures, and exercises

- If loss is NaN, inspect the number of supervised targets, precision and learning rate before adding steps.
- If every label is `-100`, repair the template/mask; an empty objective cannot teach anything.
- If frozen parameters change, inspect the trainable parameter report and optimizer parameter list.
- If tiny generations look meaningless, that is expected from random initialization and a tiny vocabulary.

**Exercises:** (1) Print which token predicts the first answer token. (2) Compare full/selective/LoRA parameter counts. (3) Change one training answer, rerun from the same seed, and inspect held-out loss. (4) Explain why saving an adapter is not enough to resume AdamW.

**Expected result:** finite objective values, some expected trainable weights changed, frozen weights unchanged, and identical greedy tokens after reload. Record actual values in `lesson_report.json`; no fixed quality threshold is asserted.


## Connect this calculation to FineTuneLab

The recipe builds the production trainer from the same canonical records. The CPU profile executes a separate one-step comparison. The real multi-model comparison is shown explicitly but is deferred to a fresh process to avoid keeping two experiments in GPU memory.


In [ ]:
METHOD = "ppo"
framework_rows = [{"prompt": r["prompt"]} for r in splits["train"][:2]]
# Map the visible experiment back to the framework boundary.
# Start a separate short run from the same base, not from the mutated notebook model.
from datasets import Dataset

from finetunelab.config import RECIPE_ADAPTER
from finetunelab.data import validate_dataset
from finetunelab.recipes import build_trainer

recipe = {
    "method": METHOD,
    "model": {
        "name_or_path": str(MODEL_PATH),
        "local_files_only": True,
        "dtype": RUNTIME.dtype,
    },
    "data": {"source": str(OUTPUT / "train.jsonl"), "max_length": 128},
    "tuning": {"strategy": "lora", "lora_rank": 4, "lora_alpha": 8},
    "training": {
        "output_dir": str(OUTPUT / "framework"),
        "max_steps": 1,
        "per_device_train_batch_size": 2,
        "gradient_accumulation_steps": 1,
        "gradient_checkpointing": False,
        "device": RUNTIME.device,
        "bf16": RUNTIME.bf16,
        "fp16": RUNTIME.fp16,
        "tf32": False,
        "logging_steps": 1,
        "save_steps": 1,
        "report_to": ["none"],
    },
    "generation": {"max_new_tokens": 4},
}
if METHOD == "ppo":
    recipe["model"]["reward_name_or_path"] = str(OUTPUT / "reward")
    recipe["model"]["value_name_or_path"] = str(OUTPUT / "value_initial")
    recipe["num_ppo_epochs"] = 1
config = RECIPE_ADAPTER.validate_python(recipe)
framework_dataset = Dataset.from_list(framework_rows)
print(validate_dataset(framework_dataset, config))
# True in CPU acceptance tests. Real multi-model runs need sufficient GPU memory.
RUN_FRAMEWORK_COMPARISON = MODE == "tiny_cpu"
if RUN_FRAMEWORK_COMPARISON:
    trainer = build_trainer(config, framework_dataset)
    framework_result = trainer.train()
    trainer.save_model(str(OUTPUT / "framework" / "final"))
    print(framework_result.metrics)
    assert all(
        math.isfinite(float(v))
        for v in framework_result.metrics.values()
        if isinstance(v, (int, float))
    )
    del trainer
else:
    print(
        "Configuration validated. In a fresh process run "
        "build_trainer(config, framework_dataset).train()."
    )

## Compare the reward stage with RewardTrainer

The earlier scalar-head update exposes the Bradley–Terry loss. This separate tiny run exercises the framework's reward-model stage as well as the PPO stage above. A real run should use a fresh process to release the policy/reference/value models first.


In [ ]:
reward_recipe = copy.deepcopy(recipe)
reward_recipe["method"] = "reward"
reward_recipe.pop("num_ppo_epochs", None)
reward_recipe["model"]["reward_name_or_path"] = str(MODEL_PATH)
reward_recipe["model"]["value_name_or_path"] = None
reward_recipe["training"]["output_dir"] = str(OUTPUT / "framework_reward")
reward_rows = [
    {"prompt": row["prompt"], "chosen": row["chosen"], "rejected": row["rejected"]}
    for row in splits["train"][:2]
]
if RUN_FRAMEWORK_COMPARISON:
    reward_trainer = build_trainer(
        RECIPE_ADAPTER.validate_python(reward_recipe), Dataset.from_list(reward_rows)
    )
    result = reward_trainer.train()
    reward_trainer.save_model(str(OUTPUT / "framework_reward" / "final"))
    assert math.isfinite(result.metrics["train_loss"])
    print(result.metrics)
    del reward_trainer